# Inference
> Load finetuned model and infer on single image

In [ ]:
#| default_exp inference.decoder_inference

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
!which python

In [ ]:
#| export
import sys
from pathlib import Path
import os
from fastcore.all import *
import shutil


In [ ]:
#| export
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_fine_tune/nbs'

In [ ]:
Path.cwd()

In [ ]:
##os.chdir('/home/hpc_storage/data/projects/userA/sam3/sam3/')
#import os
#os.chdir(CURRENT_NB)

In [ ]:
path_dir = Path(r'/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune')
print(f'path exists: {path_dir.exists()}')

In [ ]:
ckpnt = Path(path_dir, 'experiments/5cat_matrix_chip_v2/checkpoints/checkpoint_5.pt')

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
def load_model(bpe_path: str, sam3_checkpoint: Path, finetuned_checkpoint: Path, 
               device: torch.device = None) -> torch.nn.Module:
    """Load SAM3 model with fine-tuned weights.
     Args:
         bpe_path: Path to the BPE tokenizer file.
         sam3_checkpoint: Path to the original SAM3 model checkpoint.
         finetuned_checkpoint: Path to the fine-tuned model checkpoint.
         device: Optional torch device to load the model on. If None, it will automatically use CUDA if available.
     Returns:
         Loaded SAM3 model with fine-tuned weights."""
    
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = build_sam3_image_model(bpe_path=bpe_path, checkpoint_path=sam3_checkpoint)
    finetuned_weights = torch.load(finetuned_checkpoint, map_location=device, )
    model.load_state_dict(finetuned_weights['model'])
    model.eval()
    
    print("Model loaded")
    return model

In [ ]:
def run_inference(model, image_path, prompt):
    image = Image.open(image_path)
    width, height = image.size
    processor = Sam3Processor(model, confidence_threshold=0.5)
    inference_state = processor.set_image(image)
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    img0 = Image.open(image_path)
    plot_results(img0, inference_state)

In [ ]:
experiment_name = '8cat_bentwire_separate'
experiment_name = '9cat_unique_panda'
#ckpnt = Path('/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune', f'experiments/{experiment_name}/checkpoints/val_pin_detection_coco_eval_segm_AP.pt')
ckpnt = Path('/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune', f'experiments/{experiment_name}/checkpoints/checkpoint_10.pt')

In [ ]:
bpe_path = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
sam3_checkpoint = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
finetuned_checkpoint = Path("/home/hpc_storage/data/projects/userA/sam3_data/experiments/testing_3pins_1crack/checkpoints/checkpoint_10.pt")
image_path = '/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/source_images/8_In_18_0_1.png'
model = load_model(bpe_path, sam3_checkpoint, ckpnt, device=device)


In [ ]:

run_inference(model, image_path, prompt="et_pin")

In [ ]:
new_img_path= Path(r'/home/hpc_storage/data/projects/userB/data/cegled_econo/47834/curated8bit/test/images')
new_img_path.exists()

In [ ]:
new_img_path.ls()

In [ ]:
import numpy as np
idx = np.random.randint(0, len(new_img_path.ls()) - 1)
sm_im =Path(new_img_path.ls()[idx])
run_inference(model, sm_im, prompt="matrix_chip")

In [ ]:
new_path = Path(r'/home/hpc_storage/data/projects/userB/data/cegled_econo/43967/new_16bit_processed')
new_path.exists()

In [ ]:
idx = np.random.randint(0, len(new_path.ls()) - 1)
sm_nw_p = Path(new_path.ls()[idx])
run_inference(model, sm_nw_p, prompt="matrix_chip")

### SAKI chips

In [ ]:
from glob import glob


In [ ]:
d_p = Path(r'/home/hpc_storage/data/projects/userB/data/SAKI_processed/')
d_p.exists()




In [ ]:

image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}
imgs = [p for p in d_p.rglob('*') if p.suffix.lower() in image_exts]

In [ ]:
len(imgs)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
idx = np.random.randint(0, len(imgs))
sm_nw_p = imgs[idx]
run_inference(model, sm_nw_p, prompt="matrix_chip")

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

def visualize_and_save(image_path, inference_state, save_path=None, binary=False):
    """
    Visualize image, masks, and overlay with consistent colors per category.
    
    Args:
        image_path: Path to the original image
        inference_state: Inference state containing predictions
        save_path: Optional path to save the figure
        binary: If True, use black/white masks only
    """
    image = np.array(Image.open(image_path).convert("RGB"))
    h, w = image.shape[:2]
    
    # Extract boxes, labels, scores from inference_state
    boxes = inference_state.get('pred_boxes', [])
    labels = inference_state.get('pred_labels', [])
    scores = inference_state.get('pred_scores', [])
    masks = inference_state.get('pred_masks', None)
    
    # Build color map per unique category
    unique_labels = list(set(labels)) if labels else []
    if binary:
        color_map = {lbl: (1.0, 1.0, 1.0) for lbl in unique_labels}
    else:
        palette = list(mcolors.TABLEAU_COLORS.values())
        color_map = {lbl: mcolors.to_rgb(palette[i % len(palette)]) 
                     for i, lbl in enumerate(unique_labels)}

    # Create composite mask overlay
    overlay = image.copy().astype(np.float32)
    mask_canvas = np.zeros((h, w, 3), dtype=np.float32)

    if masks is not None:
        for i, (mask, label) in enumerate(zip(masks, labels)):
            color = color_map.get(label, (1.0, 1.0, 1.0))
            for c in range(3):
                mask_canvas[:, :, c] += mask * color[c]
            # Blend into overlay
            alpha = 0.4
            for c in range(3):
                overlay[:, :, c] = np.where(
                    mask > 0.5,
                    overlay[:, :, c] * (1 - alpha) + color[c] * 255 * alpha,
                    overlay[:, :, c]
                )

    mask_canvas = np.clip(mask_canvas, 0, 1)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask_canvas if not binary else mask_canvas.mean(axis=2), 
                   cmap='gray' if binary else None)
    axes[1].set_title("Masks")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    # Draw boxes
    for box, label, score in zip(boxes, labels, scores):
        color = color_map.get(label, (1.0, 1.0, 1.0))
        x1, y1, x2, y2 = box
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color, facecolor='none')
        axes[2].add_patch(rect)
        axes[2].text(x1, y1 - 5, f"{label} {score:.2f}", color=color,
                     fontsize=8, backgroundcolor='black')
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    # Legend
    legend_patches = [mpatches.Patch(color=color_map[lbl], label=lbl) 
                      for lbl in unique_labels]
    if legend_patches:
        axes[2].legend(handles=legend_patches, loc='lower right', fontsize=8)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"Saved to {save_path}")

    plt.show()
    return fig


In [ ]:
 data_path = Path(r'/home/hpc_storage/data/projects/userB/data/universal_model/ThinWire_Segmentation/train_images_18122025_thinwire_split/test')

In [ ]:

# Example usage:
visualize_and_save(sm_im, inference_state, save_path=None, binary=False)

In [ ]:
run_inference(model, sm_im, prompt="matrix_chip")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()